In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def multistage_decimation_simulation(M1, M2):
    clear_output(wait=True)
    
    M_total = M1 * M2
    
    # 1. Frequency vector covering [-1.5*pi, 1.5*pi]
    N_fft = 4000
    w = np.linspace(-1.5 * np.pi, 1.5 * np.pi, N_fft)
    
    # Base Signal Spectrum X(e^{j\omega}): Triangular spectrum with narrow bandwidth to show anti-aliasing filtering
    omega_N = 0.3 * np.pi
    
    def base_spectrum(w_axis):
        spec = np.zeros_like(w_axis)
        mask = np.abs(w_axis) <= omega_N
        spec[mask] = 1.0 - np.abs(w_axis[mask]) / omega_N
        return spec

    # Original Spectrum
    x_spec = base_spectrum(w)
    
    # --- STAGE 1: Decimation by M1 ---
    wc1 = np.pi / M1
    h1_spec = np.where(np.abs(w) <= wc1, 1.0, 0.0)
    filtered_1 = x_spec * h1_spec
    
    y1_spec = np.zeros_like(w)
    for k in range(-5, 6):
        y1_spec += (1.0 / M1) * np.interp((w - 2.0 * np.pi * k) / M1, w, filtered_1, left=0, right=0)
        
    # --- STAGE 2: Decimation by M2 (applied to Stage 1 output) ---
    wc2 = np.pi / M2
    h2_spec = np.where(np.abs(w) <= wc2, 1.0, 0.0)
    filtered_2 = y1_spec * h2_spec
    
    y2_spec = np.zeros_like(w)
    for k in range(-5, 6):
        y2_spec += (1.0 / M2) * np.interp((w - 2.0 * np.pi * k) / M2, w, filtered_2, left=0, right=0)

    # --- SINGLE-STAGE COMPARISON (Decimation by M_total) ---
    wc_total = np.pi / M_total
    h_total_spec = np.where(np.abs(w) <= wc_total, 1.0, 0.0)
    filtered_total = x_spec * h_total_spec
    
    y_total_spec = np.zeros_like(w)
    for k in range(-5, 6):
        y_total_spec += (1.0 / M_total) * np.interp((w - 2.0 * np.pi * k) / M_total, w, filtered_total, left=0, right=0)

    # --- Plotting ---
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # Subplot 1: Stage 1
    axes[0].plot(w / np.pi, x_spec, color='tab:blue', lw=2, label=r'Original Spectrum $\mathcal{X}(e^{j\omega})$')
    axes[0].plot(w / np.pi, y1_spec, color='tab:orange', lw=2.5, linestyle='-', label=r'After Stage 1 ($M_1=' + str(M1) + '$)')
    axes[0].axvline(x=wc1 / np.pi, color='green', linestyle=':', lw=1.5, label=r'Cutoff $\pi/M_1$')
    axes[0].axvline(x=-wc1 / np.pi, color='green', linestyle=':', lw=1.5)
    axes[0].set_title(r'Stage 1: Downsampling by $M_1 = ' + str(M1) + '$', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_xlim(-1.5, 1.5)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Subplot 2: Stage 2
    axes[1].plot(w / np.pi, y1_spec, color='tab:orange', lw=1.5, linestyle='--', label=r'Input to Stage 2 ($\mathcal{Y}_1$)')
    axes[1].plot(w / np.pi, y2_spec, color='tab:purple', lw=2.5, label=r'Final Multi-Stage Output ($M = M_1 \times M_2 = ' + str(M_total) + '$)')
    axes[1].axvline(x=wc2 / np.pi, color='red', linestyle=':', lw=1.5, label=r'Cutoff $\pi/M_2$')
    axes[1].axvline(x=-wc2 / np.pi, color='red', linestyle=':', lw=1.5)
    axes[1].set_title(r'Stage 2: Downsampling by $M_2 = ' + str(M2) + '$ (Total Factor $M = ' + str(M_total) + '$)', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=9)
    axes[1].set_xlim(-1.5, 1.5)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Subplot 3: Equivalence Verification
    axes[2].plot(w / np.pi, y2_spec, color='tab:purple', lw=2.5, label=r'Multi-Stage Result ($M_1 \times M_2$)')
    axes[2].plot(w / np.pi, y_total_spec, color='black', lw=1.5, linestyle='--', label=r'Single-Stage Equivalent ($M=' + str(M_total) + '$)')
    axes[2].set_title(r'Equivalence Verification: Multi-Stage vs Single-Stage ($M=' + str(M_total) + '$)', fontsize=10, fontweight='bold')
    axes[2].set_xlabel(r'Normalized Frequency ($\omega / \pi$)', fontsize=9)
    axes[2].set_ylabel('Amplitude', fontsize=9)
    axes[2].set_xlim(-1.5, 1.5)
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Conclusion Box displayed under the plots
    display(HTML(f"""
    <div style="border: 1px solid #ccc; padding: 12px; border-radius: 5px; background-color: #f9f9f9; font-family: sans-serif; font-size: 14px;">
        <strong>Conclusion:</strong> 
        The plots above demonstrate the equivalence between a <strong>multi-stage downsampling system</strong> (composed of two sequential stages with factors $M_1 = {M1}$ and $M_2 = {M2}$) and a <strong>single-stage downsampling system</strong> with a total factor of $M = {M_total}$. 
        As verified in the third subplot, both approaches yield identical final spectra, confirming that breaking down the process into multiple stages achieves significant computational and hardware efficiency without altering the theoretical outcome.
    </div>
    """))

# Interactive Sliders for M1 and M2
m1_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description='Stage 1 ($M_1$):', style={'description_width': 'initial'})
m2_slider = widgets.IntSlider(value=2, min=1, max=4, step=1, description='Stage 2 ($M_2$):', style={'description_width': 'initial'})

ui = widgets.VBox([m1_slider, m2_slider])
display(ui)

out = widgets.interactive_output(multistage_decimation_simulation, {'M1': m1_slider, 'M2': m2_slider})
display(out)